## 1. Setup and Imports

In [ ]:
from sage.all import *
import pandas as pd
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv('../data/bitcoin_timeseries.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').set_index('timestamp')


## 2. Select and Normalize Subset for Modeling (14 Days)

In [ ]:
# Use last 14 days of hourly-resampled data
df_model = df.last('14D').resample('1h').mean().dropna().reset_index()
df_model = df_model.tail(100)  # use last 100 points for stability

# Convert timestamps to numeric hours and normalize around mean
x_vals = [(t - df_model['timestamp'][0]).total_seconds() / 3600 for t in df_model['timestamp']]
x_mean = sum(x_vals) / len(x_vals)
x_vals = [x - x_mean for x in x_vals]
y_vals = list(df_model['price_usd'])

print(f"✅ Using {len(x_vals)} normalized data points")


## 3. Symbolic Least Squares Polynomial Fit (Degree 3)

In [ ]:
# Degree-3 polynomial
x = var('x')
coeffs = var('a0 a1 a2 a3')
model = coeffs[0] + coeffs[1]*x + coeffs[2]*x**2 + coeffs[3]*x**3

# Residuals and squared error
residuals = [model.subs(x=x_i) - y_i for x_i, y_i in zip(x_vals, y_vals)]
squared_error = sum(r**2 for r in residuals)

# Minimize error
solution = minimize(squared_error, coeffs)
model_fitted = model.subs(dict(zip(coeffs, solution)))
model_fitted


## 4. Plot Fitted Polynomial vs Actual

In [ ]:
f = lambda t: float(model_fitted.subs(x=t))
y_fit = [f(t) for t in x_vals]

plt.figure(figsize=(12, 5))
plt.plot(df_model['timestamp'], y_vals, label='Actual Price', marker='o')
plt.plot(df_model['timestamp'], y_fit, label='Fitted Polynomial (Least Squares)', linestyle='--')
plt.xlabel("Time")
plt.ylabel("Price (USD)")
plt.title("BTC Price - Stable Symbolic Polynomial Fit (14 Days)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('../reports/symbolic_polynomial_fit_stable.png')
plt.show()
